# Runner — Anatomy-Aware Pose Estimation (STL Training)
Notebook **minimale**: tutto il codice vero sta nei moduli `.py` sul repo GitHub.

**Prima di lanciare:**
1. Settings → **Internet: On**
2. **Add Input** → COCO 2017 Keypoints + OCHuman + dataset `pose-baseline-checkpoint`

In [ ]:
# === 1. Codice dal repo GitHub ===
import os, sys

REPO_URL = "https://github.com/flaviomassaroni/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks.git"
REPO_DIR = "/kaggle/working/repo"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

mod_dir = None
for root, dirs, files in os.walk(REPO_DIR):
    if ".git" in root: continue
    if "config.py" in files:
        mod_dir = root
        break
if mod_dir:
    sys.path.insert(0, mod_dir)
    print("Moduli:", sorted([f for f in os.listdir(mod_dir) if f.endswith(".py")]))
else:
    print("ERRORE: config.py non trovato!")

In [ ]:
# === 2. Import + seed + check ===
from config import *
import utils, data, network, train
import evaluation as ev
from stl import SkeletalTopologyLoss

set_seed(SEED)
print("Device:", device)
print("\n/kaggle/input contiene:")
for d in os.listdir("/kaggle/input"):
    print("  -", d)

In [ ]:
# === 3. Dati COCO ===
from torch.utils.data import DataLoader

train_samples = data.build_samples(COCO_TRAIN_ANN, min_keypoints=8)
val_samples   = data.build_samples(COCO_VAL_ANN)
print(f"train: {len(train_samples)} | val: {len(val_samples)}")

train_dataset = data.COCOKeypointsDataset(train_samples, COCO_TRAIN_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
val_dataset   = data.COCOKeypointsDataset(val_samples,   COCO_VAL_IMG,   INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)

In [ ]:
# === 3b. Trova il best.pth della baseline ===
import subprocess
result = subprocess.run(["find", "/kaggle/input", "-name", "best.pth"], capture_output=True, text=True)
BEST_PTH = result.stdout.strip().split("\n")[0]
print("Checkpoint baseline:", BEST_PTH)
assert BEST_PTH and os.path.exists(BEST_PTH), "best.pth non trovato! Aggiungi pose-baseline-checkpoint come Input."

In [ ]:
# === 4. Fine-tuning con STL ===
import torch, os, time
from tqdm.auto import tqdm

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(BEST_PTH, map_location=device))
print("Pesi baseline caricati")

NUM_EPOCHS_STL = 15
LR_STL = 1e-4
LAMBDA_BONE, LAMBDA_ANGLE, LAMBDA_ORDER = 0.5, 0.5, 0.5

optimizer = torch.optim.Adam(model.parameters(), lr=LR_STL)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10], gamma=0.1)
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=LAMBDA_BONE, lambda_angle=LAMBDA_ANGLE,
    lambda_order=LAMBDA_ORDER, beta=10.0,
)

STL_CKPT_DIR = '/kaggle/working/checkpoints_stl'
os.makedirs(STL_CKPT_DIR, exist_ok=True)
best_val = float('inf')

for epoch in range(1, NUM_EPOCHS_STL + 1):
    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ['heatmap','bone','angle','order','total']}
    for imgs, hms, w in tqdm(train_loader, desc=f"train {epoch}", leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums: sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader.dataset)
    for k in sums: sums[k] /= n

    model.eval()
    val_sum = 0
    with torch.no_grad():
        for imgs, hms, w in tqdm(val_loader, desc=f"val {epoch}", leave=False):
            imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
            loss, _ = criterion(model(imgs), hms, w)
            val_sum += loss.item() * imgs.size(0)
    val_loss = val_sum / len(val_loader.dataset)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    print(f"E{epoch:02d} tot:{sums['total']:.4f} hm:{sums['heatmap']:.4f} "
          f"bone:{sums['bone']:.4f} ang:{sums['angle']:.4f} ord:{sums['order']:.4f} "
          f"val:{val_loss:.4f} lr:{lr:.0e} {time.time()-t0:.0f}s")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), f'{STL_CKPT_DIR}/best_stl.pth')
        print(f"  -> nuovo best STL")

print("\nFine-tuning completato.")

In [ ]:
# === 5. Valutazione: confronto baseline vs STL ===
import torch

model_base = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_base.load_state_dict(torch.load(BEST_PTH, map_location=device))
print("=== BASELINE ===")
coco_base, avr_coco_base = ev.evaluate_on_coco_val(model_base, val_samples, device)
oc_base, avr_oc_base = ev.evaluate_on_ochuman(model_base, device)

model_stl = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_stl.load_state_dict(torch.load(f'{STL_CKPT_DIR}/best_stl.pth', map_location=device))
print("\n=== STL ===")
coco_stl, avr_coco_stl = ev.evaluate_on_coco_val(model_stl, val_samples, device,
    results_path='/kaggle/working/coco_val_pred_stl.json')
oc_stl, avr_oc_stl = ev.evaluate_on_ochuman(model_stl, device,
    results_path='/kaggle/working/ochuman_pred_stl.json')

LABELS = ["AP","AP.50","AP.75","AP_M","AP_L","AR","AR.50","AR.75","AR_M","AR_L"]
header = f"{'':15}{'COCO base':>10}{'COCO STL':>10}{'':3}{'OC base':>10}{'OC STL':>10}"
print(f"\n{header}")
print("-" * 58)
for i, lab in enumerate(LABELS):
    cb = coco_base.stats[i]; cs = coco_stl.stats[i]
    ob = oc_base.stats[i]; os_ = oc_stl.stats[i]
    print(f"{lab:15}{cb:10.4f}{cs:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")
ab = avr_coco_base['AVR_pose_rate']; as_ = avr_coco_stl['AVR_pose_rate']
ob = avr_oc_base['AVR_pose_rate']; os_ = avr_oc_stl['AVR_pose_rate']
print(f"\n{'AVR rate':15}{ab:10.4f}{as_:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")
ab = avr_coco_base['AVR_mean_violations']; as_ = avr_coco_stl['AVR_mean_violations']
ob = avr_oc_base['AVR_mean_violations']; os_ = avr_oc_stl['AVR_mean_violations']
print(f"{'AVR mean':15}{ab:10.4f}{as_:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")